<a href="https://www.kaggle.com/code/jatin2055/langsmith-rag-v1?scriptVersionId=265053283" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [138]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [193]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader  
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.messages import SystemMessage, HumanMessage

import os

In [140]:
api_key='your_api_key'

langsmith = 'your_api_key'

os.environ["LANGCHAIN_API_KEY"] = langsmith
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"]="LangSmith - RAG v1"

os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'


In [141]:
model = ChatOpenAI(api_key=api_key)

# LOADER

In [142]:
PDF_PATH = ["https://www.stat.berkeley.edu/~rabbee/s154/ISLR_First_Printing.pdf", "https://www.sas.upenn.edu/~fdiebold/NoHesitations/BookAdvanced.pdf"]

#"https://www.sas.upenn.edu/~fdiebold/NoHesitations/BookAdvanced.pdf"

In [143]:
all_documents = []

for path in PDF_PATH:
    print(path)
    loader = PyPDFLoader(path)
    all_documents.extend(loader.lazy_load())

https://www.stat.berkeley.edu/~rabbee/s154/ISLR_First_Printing.pdf
https://www.sas.upenn.edu/~fdiebold/NoHesitations/BookAdvanced.pdf


In [144]:
print(len(all_documents))

1205


# Creating Chunks

In [145]:
splitter = RecursiveCharacterTextSplitter(chunk_size =800, chunk_overlap=100)

pdf_split = splitter.split_documents(all_documents)

In [146]:
print(len(pdf_split))


4532


# Embedding

In [147]:
embed = OpenAIEmbeddings(model='text-embedding-3-small', api_key=api_key)
vector_store = FAISS.from_documents(pdf_split, embed)

retriever = vector_store.as_retriever(
    search_type="mmr", # Maximum Marginal Relevance 
    search_kwargs = {"k": 3, "lambda_mult": 0.5 } # lambda_mult [0-1] : 0 - most diverse, 1: most similar (same as similarity search) 
)

In [148]:
#result = retriever.invoke("confusion matrix")


In [149]:
# for r in result:
#     print(r)
#     print('*'*30)

# Prompt

In [205]:
# This works
message = [
    ("system","""Answer only from the provided context. If not found, reply back You dont know"""),
    ("human",""" question: {question}.\n\n context : {context}""")
]

# OR 
# This doesnot works - dont know why?
# message = [
#     SystemMessage(content="""Answer only from the provided context. If not found, reply back You dont know"""),
#     HumanMessage(content=""" question: {question}.\n\n context : {context}""")
# ]


In [206]:
prompt = ChatPromptTemplate.from_messages(message)

In [207]:
def combine_retrieved_docs(embed_docs):
    return "\n".join(d.page_content for d in embed_docs)

In [208]:
parallel = RunnableParallel({
    "context": retriever | RunnableLambda(combine_retrieved_docs),
    "question": RunnablePassthrough()
    
})

# Output Parser

In [209]:
str_parser = StrOutputParser()

# Chaining

In [210]:
chain = parallel | prompt | model | str_parser

In [211]:
config = {
    'run_name': 'LangSmith - RAG V1',
    'tags':['ChatPromptTemplate', 'FAISS', 'retriever', "mmr", "RecursiveCharacterTextSplitter", "PyPDFLoader", "load", "lazy_load"],
    'metadata': {'model': 'openai', 'parser_used': 'str output'}
}

In [212]:
q = input("Ask you question: ")
print(q)
print(q.strip())
ans = chain.invoke(q.strip(), config=config)

Ask you question:  Name the authors


Name the authors
Name the authors


In [213]:
print(ans)

The authors are Trevor Hastie, Robert Tibshirani, and Jerome Friedman.
